In [0]:
import os
import re
import nltk
import string
import pandas as pd
from nltk.stem import PorterStemmer
import unicodedata
import csv
!pip install autocorrect
import autocorrect
from autocorrect import Speller
from nltk.corpus import *
import networkx as nx
from ast import literal_eval
from itertools import combinations

nltk.download('words')
nltk.download('wordnet')
nltk.download('punkt')

def convert_text_to_sentences(text):
    return nltk.sent_tokenize(text)

def splitting(string):
  return string.split(" ")

flat_list=[]
def removeNestings(l): 
    for i in l: 
        if type(i) == list: 
            removeNestings(i) 
        else: 
            flat_list.append(i) 


G = nx.Graph()
rdr = csv.reader(open('knoGraph.csv', 'r'))
rdr = list(rdr)
for i in rdr:
    G.add_edge(i[0], i[1], weight=float(i[2]))

rdr = csv.reader(open('index1.csv', 'r'))
rdr = list(rdr)

d = dict()

for i in rdr:
    d[i[0]] = i[1:]



text = input("Search: ")
print(" Searching :", text)

newline_ex = re.compile("\n")
numbers_ex = re.compile("[0-9]+(-[0-9]+)?")
punctuation_ex = re.compile("[^a-z]-")
roman_num_ex = re.compile("\\b[i|v|x|l|c|d|m]{1,3}\\b")
stopwords = read_list_from_file(path, "stopwords.txt")
ps = PorterStemmer()
check = Speller(lang='en')

text = re.sub(newline_ex, ' ', text)
text = unicode(text, errors='replace')
#sen = convert_text_to_sentences(text)

## pre-processing text
text = text.strip()
text = text.lower()
text = re.sub(numbers_ex, '', text)
text = re.sub(punctuation_ex, '', text)
text = re.sub(roman_num_ex, '', text)
text= text.replace("-", " ")

words= [] #isme input string aayega 
word=[] #isme sare process hoke save honge

words.append(text)

# stopword removal
words = [i for i in words if i not in stopwords]
print(words)

#spell checker
words =[check(i) for i in words]
print("Spell checker :" , words)

split_words=[]

#splitting
for i in words:
    split_words.append(splitting(i))

#converting list of list into a single list
removeNestings(split_words)

token=[]
#stemming
token = [ps.stem(i) for i in flat_list]
word.append(token)
print("Stemming : " , word)

spaceless_words=[]
#merging
spaceless_words= "".join(token)
word.append(spaceless_words)

print("Spaceless words :", word)

comb=[]
#n grams segmentation
comb=combinations(token,2)
for i in list(comb):
      word.append(i)
      
print("Segmentation: " , word)

#synonym and antonyms
synonyms = []
antonyms = []
for i in token:
    for syn in wordnet.synsets(i): 
      for l in syn.lemmas(): 
        synonyms.append(l.name()) 
        if l.antonyms(): 
            antonyms.append(l.antonyms()[0].name()) 
  
word.append(set(synonyms)) 

if len(antonyms)!=0:
  word.append(set(antonyms)) 
print("Synonyms and antonyms: ", word)

#ye graph se karne ka copied hai 
l = len(words)

q = words[0]

for i in range(1, l):
    q = q + '-' + words[i]

nbr = sorted(G.edges(nbunch=q, data=True), key=lambda t: t[2].get('weight', 1))[::-1]

for i in range(len(d[q])):
    d[q][i] = literal_eval(d[q][i])
d[q].sort(key=lambda x: x[1])

vis = []

for i in d[q][:8]:
    vis.append(i[0][:-4])
    print(i[0][:-4])

y = 8

for n in nbr[:7]:
    print(n[1])
    if n[1] not in d.keys():
        continue
    for i in range(len(d[n[1]])):
        d[n[1]][i] = literal_eval(d[n[1]][i])
    d[n[1]].sort(key=lambda x: [1])

    j = 1

    for i in d[n[1]]:
        if i[0][:-4] in vis:
            continue
        vis.append(i[0][:-4])
        print(i[0][:-4])
        j += 1
        if j >= y:
            break
    y -= 1



[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Search: good is nice
 Searching : good is nice
Spell checker : ['good is nice']
Stemming :  [['good', 'is', 'nice']]
[['good', 'is', 'nice'], 'goodisnice']
Segmentation:  [['good', 'is', 'nice'], 'goodisnice', ('good', 'is'), ('good', 'nice'), ('is', 'nice')]
Synonyms and antonyms:  [['good', 'is', 'nice'], 'goodisnice', ('good', 'is'), ('good', 'nice'), ('is', 'nice'), {'be', 'serious', 'secure', 'constitute', 'embody', 'upright', 'adept', 'near', 'skilful', 'right', 'ripe', 'personify', 'live', 'in_effect', 'honorable', 'nice', 'prissy', 'dear', 'equal', 'comprise', 'commodity', 'effective', 'unspoiled', 'thoroughly', 'sound', 'undecomposed', 'dainty', 

NameError: ignored